Project Drishti · Any data. Any domain. One grammar.

Copyright (c) 2026 Ashutosh Sinha <ajsinha@gmail.com>.
All rights reserved.

PROPRIETARY AND CONFIDENTIAL.

This file is the confidential and proprietary property of Ashutosh Sinha.
Unauthorised copying, use, modification, distribution or disclosure of this
file, via any medium, is strictly prohibited except with the express prior
written permission of the copyright holder.

See the LICENSE file in the root of this repository for the full terms.

# 02 · A kind into pandas and polars

Read the trading pack's `trade` kind (the demo lake's synthetic trades of three booking systems) as an Arrow table, then hand it to pandas and to polars without a copy through JSON. The read is the Data view's own reader: your entitlements, row rules and masks apply, and it is audited as a download.

*Read only:* this notebook only reads. Flight refuses every call that would write (`DoPut`, `DoExchange`), and nothing here asks for one.

**Setup.** `flight_setup.py` (beside this notebook) finds the one-file client `clients/python/drishti_flight.py` and reads the address, the token and the CA from the environment: `DRISHTI_FLIGHT_URL`, `DRISHTI_TOKEN` (or `DRISHTI_TOKEN_FILE`) and `DRISHTI_FLIGHT_CA`. No token is ever written in a notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))     # run Jupyter from examples/clients/python, where flight_setup.py is
from flight_setup import connect, meta, FlightRefused, SCOPE, DATE, STREAM_WAIT

## Read the chosen paths

Name the paths you want (fewer paths, a smaller read). No `date`: the latest business date the kind holds; `DRISHTI_EXAMPLE_DATE` picks another. A masked column is left out of the answer rather than sent empty.

In [ ]:
f = connect()
PATHS = ["TRADE_ID", "SOURCE_SYSTEM", "PRODUCT_TYPE", "TT_GROUP", "PORTFOLIO", "CURRENCY", "NOTIONAL", "TRADE_DATE"]
trades = f.kind("trade", paths=PATHS, date=DATE)
print(trades.num_rows, "rows")
print(trades.schema)

## pandas

`to_pandas()` converts the Arrow columns. The types follow the kind's data contract: amounts as float64, integers as int64, the rest as text.

In [ ]:
df = trades.to_pandas()
df.groupby(["SOURCE_SYSTEM", "TT_GROUP"], dropna=False)["NOTIONAL"].agg(["count", "sum"]).head(15)

## polars

`polars.from_arrow` takes the same table, zero copy where the types allow.

In [ ]:
import polars as pl

pf = pl.from_arrow(trades)
(pf.group_by("PRODUCT_TYPE")
   .agg(pl.len().alias("trades"), pl.col("NOTIONAL").sum().alias("notional"))
   .sort("notional", descending=True)
   .head(10))

## Filter on the server

`where` lines are the Data view's filter lines (`FIELD = value`, `!=`, `in`, `contains`, `starts`, `>`, `<`, `between`, `is empty`): the server reads only the rows asked for.

In [ ]:
irs = f.kind("trade", paths=["TRADE_ID", "PRODUCT_TYPE", "CURRENCY", "NOTIONAL"], where=["PRODUCT_TYPE = IRS"], date=DATE)
print(irs.num_rows, "IRS trades")
irs.to_pandas().head()

In [ ]:
f.close()